# The `this` Keyword in JavaScript

## Definition

In JavaScript, `this` refers to the object that is currently executing the function. Unlike variables, its value is not static — it is determined **dynamically at runtime** based on *how* a function is called, not *where* it is defined.

To control or maintain the reference of `this`, JavaScript uses context binding rules.

## The 4 Core Binding Rules

JavaScript determines the value of `this` by evaluating the function's invocation site against four prioritized rules.

### 1. Default Binding (Standalone Invocation)

When a regular function is called globally or on its own (e.g., `foo()`), `this` defaults to the global environment.

- **Non-strict mode:** `this` points to the `window` object (in browsers) or `global` (in Node.js).
- **Strict mode** (`'use strict';`): `this` evaluates to `undefined`.

### 2. Implicit Binding (Object Method Invocation)

When a function is called as a method of an object (using dot notation), `this` points to the object directly to the left of the dot.

```javascript
const user = {
  name: 'Alice',
  greet() {
    console.log(this.name); // 'this' points to 'user'
  }
};
user.greet(); // Outputs: Alice
```

⚠️ **Context Loss:** If you assign an implicit method to a variable and execute it later (e.g., `const launch = user.greet; launch();`), it reverts to Default Binding and loses its connection to the object.

```javascript
const launch = user.greet;
launch(); // Outputs: undefined (or throws in strict mode)
```

### 3. Explicit Binding (`call`, `apply`, `bind`)

If you want to force a function to use a specific object as its context, you can explicitly bind it using built-in methods:

- **`call()`** — Invokes the function immediately, passing the context first and subsequent arguments individually.
- **`apply()`** — Invokes the function immediately, passing the context first and arguments inside a single array.
- **`bind()`** — Does not execute the function immediately. Instead, it returns a brand new function with the `this` context permanently locked to the provided target.

```javascript
function introduce(location, era) {
  console.log(`${this.name} is in ${location}, ${era}`);
}

const character = { name: 'Doctor' };

introduce.call(character, 'London', '1963');   // Immediate execution
introduce.apply(character, ['London', '1963']); // Immediate execution

const boundFunc = introduce.bind(character);    // Returns a new function
boundFunc('Gallifrey', 'Time War');             // Executed later
```

### 4. `new` Binding (Constructor Functions)

When a function is invoked with the `new` keyword, JavaScript instantiates a blank object, and `this` inside the constructor is bound entirely to that new object.

```javascript
function Car(model) {
  this.model = model; // 'this' maps to the fresh instance
}
const myCar = new Car('Tesla');
```

Under the hood, `new` does four things: creates a new empty object, sets that object's prototype to the constructor's `.prototype`, binds `this` to it inside the function call, and returns the new object automatically (unless the constructor explicitly returns a different object).

## Lexical Binding (Arrow Functions)

[Arrow functions](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Functions/Arrow_functions) do not follow standard binding rules. They do not possess their own `this` context. Instead, they inherit `this` from their enclosing parent block scope at the moment they are written (lexical scoping).

- You cannot override an arrow function's context using `call()`, `apply()`, or `bind()`.
- They are ideal for callbacks (like `setTimeout` or array methods) because they won't lose track of the surrounding object context.

```javascript
const counter = {
  count: 0,
  start() {
    // Arrow function preserves 'this' from start() method context
    setInterval(() => {
      this.count++;
      console.log(this.count);
    }, 1000);
  }
};
```

Compare this to what would happen with a regular `function` in the same spot — the callback would lose the `counter` context entirely and `this.count` would be `undefined` (or throw), since `setInterval` invokes the callback with Default Binding:

```javascript
const counterBroken = {
  count: 0,
  start() {
    setInterval(function() {
      this.count++; // 'this' is NOT counterBroken here — Default Binding applies
      console.log(this.count); // NaN or error
    }, 1000);
  }
};
```

This distinction — "does the callback need to see the surrounding object?" — is usually the practical test for choosing an arrow function vs. a regular function for callbacks.

## Summary of Precedence

If multiple rules collide, JavaScript applies binding based on this order of priority (highest to lowest):

1. **`new` Binding**
2. **Explicit Binding** (`bind`, `call`, `apply`)
3. **Implicit Binding** (Object methods)
4. **Default Binding** (Global or `undefined`)

> Arrow functions sit outside this hierarchy entirely — they never get their own `this`, so none of the four rules can apply to them. Calling `.call()`/`.apply()`/`.bind()` on an arrow function silently has no effect on its `this`.

## Quick Recap

| Rule | Trigger | `this` resolves to |
|---|---|---|
| Default | Standalone call: `foo()` | `window`/`global`, or `undefined` in strict mode |
| Implicit | Method call: `obj.foo()` | The object left of the dot |
| Explicit | `foo.call(obj)` / `.apply(obj)` / `.bind(obj)` | The object passed in |
| `new` | `new Foo()` | The newly created instance |
| Lexical (arrow) | Arrow function `() => {}` | Whatever `this` was in the enclosing scope |

## Sources

1. [MDN — `this`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/this)
2. [GeeksforGeeks — JavaScript `this` keyword](https://www.geeksforgeeks.org/javascript/javascript-this-keyword/)
3. [Medium — `this` Keyword and Bindings in JavaScript](https://medium.com/@supraja_miryala/this-keyword-and-bindings-in-javascript-176534e3dd32)
4. [Codementor — What's the binding of `this`?](https://www.codementor.io/@diegopalacios/what-s-the-binding-of-this-1agz39841z)
5. [Gist — `this` binding cheatsheet](https://gist.github.com/zcaceres/2a4ac91f9f42ec0ef9cd0d18e4e71262)
6. [GeeksforGeeks — JavaScript Function Binding](https://www.geeksforgeeks.org/javascript/javascript-function-binding/)
7. [freeCodeCamp — JavaScript `this` Keyword Binding Rules](https://www.freecodecamp.org/news/javascript-this-keyword-binding-rules/)
8. [MDN — Arrow function expressions](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Functions/Arrow_functions)